# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the Olist product category name translation dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports product and category analysis by providing standardized product category names and their English translations.

### **Import The Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### **Load The Dataset**

In [2]:
category_df = pd.read_csv (r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\product_category_name_translation.csv")

## **PRODUCT NAME CATEGORY TRANSLATION DATASET**

### **1. Data Inpection**

In [ ]:
# The first 5 records of the dataset
category_df.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [7]:
# The number of rows and columns in the dataset
category_df.shape

(71, 2)

In [8]:
# The names of the columns and their datatypes
category_df.dtypes

product_category_name            object
product_category_name_english    object
dtype: object

The data types are appropriate for the structure of the dataset. Both `product_category_name` and `product_category_name_english` are stored as **`object`**, which is suitable for categorical text values.

In [9]:
# The number of missing values in the dataset
category_df.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

No missing values were identified. Both `product_category_name` and `product_category_name_english` contain **0 missing values** across all **71 records**, indicating that the dataset is complete for these fields. No cleaning is required for missing values.

In [ ]:
# The number of duplicated values in the dataset
category_df.duplicated().sum()

np.int64(0)

In [13]:
# The number of duplicated values in each column
for column in category_df.columns:
    duplicates = category_df[column].duplicated().sum()
    print(f"{column}: {duplicates}")

product_category_name: 0
product_category_name_english: 0


No duplicated values were identified. Both `product_category_name` and `product_category_name_english` contain **0 duplicated values**, with each category occurring only once in its respective column. No cleaning is required for duplicated values.

In [15]:
# Summary statitics of the dataset
category_df.describe()

,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


The results are consistent with the structure of the dataset. Both `product_category_name` and `product_category_name_english` contain **71 values** and **71 unique values**, while the highest frequency is **1** for both columns. This confirms that every category occurs once and is consistent with the absence of duplicates. As both columns contain categorical text data

### **2. Data Profiling** 

#### **Category-name structure**

In [22]:
# The length distribution of product category names
category_df["product_category_name"].str.len().describe()

count    71.000000
mean     16.971831
std       8.325472
min       3.000000
25%      11.000000
50%      16.000000
75%      22.000000
max      46.000000
Name: product_category_name, dtype: float64

In [21]:
# The length distribution of English product category names
category_df["product_category_name_english"].str.len().describe()

count    71.000000
mean     16.070423
std       7.838229
min       3.000000
25%      10.000000
50%      15.000000
75%      21.000000
max      39.000000
Name: product_category_name_english, dtype: float64

In [23]:
# The shortest and longest product category names
category_df.loc[
    category_df["product_category_name"].str.len().isin(
        [
            category_df["product_category_name"].str.len().min(),
            category_df["product_category_name"].str.len().max()
        ]
    ),
    ["product_category_name"]
]

,product_category_name
26,moveis_cozinha_area_de_servico_jantar_e_jardim
53,pcs


In [24]:
# The shortest and longest English product category names
category_df.loc[
    category_df["product_category_name_english"].str.len().isin(
        [
            category_df["product_category_name_english"].str.len().min(),
            category_df["product_category_name_english"].str.len().max()
        ]
    ),
    ["product_category_name_english"]
]

,product_category_name_english
26,kitchen_dining_laundry_garden_furniture
46,art


The category names show a reasonable range of lengths. `product_category_name` ranges from **3 to 46 characters**, while `product_category_name_english` ranges from **3 to 39 characters**. The mean and median lengths are relatively close for both columns, and the observed minimum and maximum values do not indicate an obvious data-quality issue based on length alone

#### **Whitespace and formatting**

In [25]:
# The number of leading and trailing whitespace values
category_df[
    category_df["product_category_name"].str.strip()
    != category_df["product_category_name"]
].shape[0]

0

In [26]:
# The number of leading and trailing whitespace values
category_df[
    category_df["product_category_name_english"].str.strip()
    != category_df["product_category_name_english"]
].shape[0]

0

In [27]:
# The number of values containing multiple consecutive spaces
category_df["product_category_name"].str.contains(r"\s{2,}", regex=True).sum()

np.int64(0)

In [28]:
# The number of values containing multiple consecutive spaces
category_df["product_category_name_english"].str.contains(r"\s{2,}", regex=True).sum()

np.int64(0)

No leading or trailing whitespace was identified. Both `product_category_name` and `product_category_name_english` contain **0 values** with leading or trailing whitespace

No multiple consecutive spaces were identified. Both `product_category_name` and `product_category_name_english` contain **0 values** with consecutive whitespace.

#### **Character and punctuation consistency**

In [29]:
# The special characters and punctuation used in product category names
category_df["product_category_name"].str.findall(r"[^\w\s]").explode().value_counts()

Series([], Name: count, dtype: int64)

In [30]:
# The special characters and punctuation used in English product category names
category_df["product_category_name_english"].str.findall(r"[^\w\s]").explode().value_counts()

Series([], Name: count, dtype: int64)

In [31]:
# The category names containing repeated punctuation or special characters
category_df[
    category_df["product_category_name"].str.contains(r"[^\w\s].*[^\w\s]", regex=True)
]["product_category_name"]

Series([], Name: product_category_name, dtype: object)

In [32]:
# The English category names containing repeated punctuation or special characters
category_df[
    category_df["product_category_name_english"].str.contains(r"[^\w\s].*[^\w\s]", regex=True)
]["product_category_name_english"]

Series([], Name: product_category_name_english, dtype: object)

No special characters or punctuation were identified in either `product_category_name` or `product_category_name_english`. Both columns returned **0 identified occurrences**, so no punctuation-related

No values containing repeated punctuation or special characters were identified. Both `product_category_name` and `product_category_name_english` returned **0 values**

#### **Unicode and Encoding Quality**

In [33]:
# The category names containing accented characters
category_df[
    category_df["product_category_name"].str.contains(
        r"[áàãâäéèêëíìîïóòõôöúùûüç]",
        case=False,
        regex=True
    )
]["product_category_name"]

Series([], Name: product_category_name, dtype: object)

In [34]:
# The category names containing accented characters
category_df[
    category_df["product_category_name"].str.contains(
        r"[áàãâäéèêëíìîïóòõôöúùûüç]",
        case=False,
        regex=True
    )
]["product_category_name"]

Series([], Name: product_category_name, dtype: object)

In [35]:
# The category names containing common encoding artefacts
category_df[
    category_df["product_category_name"].str.contains(
        r"Ã|Â|â|�",
        regex=True
    )
]["product_category_name"]

Series([], Name: product_category_name, dtype: object)

In [36]:
# The English category names containing common encoding artefacts
category_df[
    category_df["product_category_name_english"].str.contains(
        r"Ã|Â|â|�",
        regex=True
    )
]["product_category_name_english"]

Series([], Name: product_category_name_english, dtype: object)

No accented characters were identified in either `product_category_name` or `product_category_name_english`. Both checks returned **0 values**, so no issue was identified from the accented-character check.

No common encoding artefacts such as `Ã`, `Â`, `â`, or `�` were identified in either column. Both checks returned **0 values**, indicating no evidence of encoding corruption. No cleaning is required.

#### **Case consistency**

In [40]:
# The capitalization patterns of product category names
category_df["product_category_name"].apply(
    lambda x: "lowercase" if x.islower()
    else "uppercase" if x.isupper()
    else "mixed"
).value_counts()


product_category_name
lowercase    71
Name: count, dtype: int64

In [38]:
# The capitalization patterns of English product category names
category_df["product_category_name_english"].apply(
    lambda x: "lowercase" if x.islower()
    else "uppercase" if x.isupper()
    else "mixed"
).value_counts()

product_category_name_english
lowercase    71
Name: count, dtype: int64

In [41]:
# The product category names with uppercase or mixed capitalization
category_df[
    ~category_df["product_category_name"].str.islower()
]["product_category_name"]

Series([], Name: product_category_name, dtype: object)

In [42]:
# The English product category names with uppercase or mixed capitalization
category_df[
    ~category_df["product_category_name_english"].str.islower()
]["product_category_name_english"]

Series([], Name: product_category_name_english, dtype: object)

All **71 values** in both `product_category_name` and `product_category_name_english` are lowercase, with **0 uppercase values** identified in either column. The capitalization format is therefore consistent across the dataset

#### **Category Naming Consistency**

In [43]:
# The separators used in product category names
category_df["product_category_name"].str.findall(r"[-_/]").explode().value_counts()

product_category_name
_    94
Name: count, dtype: int64

In [44]:
# The separators used in English product category names
category_df["product_category_name_english"].str.findall(r"[-_/]").explode().value_counts()

product_category_name_english
_    91
Name: count, dtype: int64

In [45]:
# The product category names containing separators
category_df[
    category_df["product_category_name"].str.contains(r"[-_/]", regex=True)
]["product_category_name"]

0                                       beleza_saude
1                             informatica_acessorios
3                                    cama_mesa_banho
4                                   moveis_decoracao
5                                      esporte_lazer
7                              utilidades_domesticas
9                                 relogios_presentes
10                                 alimentos_bebidas
13                          tablets_impressao_imagem
15                                    telefonia_fixa
16                                ferramentas_jardim
17                       fashion_bolsas_e_acessorios
19                                    consoles_games
21                                  fashion_calcados
22                                        cool_stuff
23                                  malas_acessorios
25                 construcao_ferramentas_construcao
26    moveis_cozinha_area_de_servico_jantar_e_jardim
27                     construcao_ferramentas_

In [46]:
# The English product category names containing separators
category_df[
    category_df["product_category_name_english"].str.contains(r"[-_/]", regex=True)
]["product_category_name_english"]

0                               health_beauty
1                       computers_accessories
3                              bed_bath_table
4                             furniture_decor
5                              sports_leisure
9                               watches_gifts
10                                 food_drink
13                     tablets_printing_image
15                            fixed_telephony
16                               garden_tools
17                   fashion_bags_accessories
18                           small_appliances
19                             consoles_games
21                              fashion_shoes
22                                 cool_stuff
23                        luggage_accessories
24                           air_conditioning
25            construction_tools_construction
26    kitchen_dining_laundry_garden_furniture
27                   costruction_tools_garden
28                      fashion_male_clothing
29                                

All **71 values** in both `product_category_name` and `product_category_name_english` contain at least one separator. `product_category_name` contains **94 separator occurrences**, while `product_category_name_english` contains **91**. The presence of separators across all records indicates that they form part of the category naming structure rather than isolated formatting issues.

#### **Category Naming Consistency**

In [48]:
# The types and number of separators used in product category names
category_df["product_category_name"].str.findall(r"[-_/]").explode().value_counts()

product_category_name
_    94
Name: count, dtype: int64

In [49]:
# The types and number of separators used in English product category names
category_df["product_category_name_english"].str.findall(r"[-_/]").explode().value_counts()

product_category_name_english
_    91
Name: count, dtype: int64

Only the underscore (`_`) separator was identified in both `product_category_name` and `product_category_name_english`. There were **94 underscore occurrences** in `product_category_name` and **91 underscore occurrences** in `product_category_name_english`. No hyphens or slashes were identified, indicating a consistent separator convention across both columns.

#### **Relationship With Products**

In [50]:
products_df = pd.read_csv (r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\cleaned\olist_products_cleaned.csv")

In [51]:
# The product categories in products that are not in the translation dataset
products_df.loc[
    ~products_df["product_category_name"].isin(category_df["product_category_name"]),
    "product_category_name"
].dropna().unique()

array(['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
      dtype=object)

In [52]:
# The number of product categories in products without a translation
products_df.loc[
    ~products_df["product_category_name"].isin(category_df["product_category_name"]),
    "product_category_name"
].dropna().nunique()

2

In [53]:
# The translation categories that are not referenced in the products dataset
category_df.loc[
    ~category_df["product_category_name"].isin(products_df["product_category_name"]),
    "product_category_name"
]

Series([], Name: product_category_name, dtype: object)

In [54]:
# The number of shared product categories between the two datasets
products_df["product_category_name"].dropna().isin(
    category_df["product_category_name"]
).sum()

np.int64(32328)

In [55]:
# The number of product records for each category without a translation
products_df[
    products_df["product_category_name"].isin(
        ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"]
    )
]["product_category_name"].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

In [56]:
# The number of product records with a non-null category
products_df["product_category_name"].notna().sum()

np.int64(32341)

The translation dataset covers **32,328 of 32,341** product records with a non-null `product_category_name`, representing approximately **99.96%** of the records. **13 records (0.04%)** are not covered by the translation dataset, belonging to `portateis_cozinha_e_preparadores_de_alimentos` (**10 records**) and `pc_gamer` (**3 records**). This indicates a minor translation coverage gap affecting **2 product categories**. No source values will be altered or new translations added without sufficient evidence.

### **3. Data Cleaning**

No cleaning will be performed. We preserve the translation dataset exactly as provided.

The 13 unmatched product records are documented as a coverage issue, not corrected, because we don't have sufficient evidence to modify the source translation dataset.

In [57]:
# Final validation of the cleaned dataset
print("Shape:", category_df.shape)
print("\nColumns:", category_df.columns.tolist())
print("\nData types:")
print(category_df.dtypes)
print("\nMissing values:")
print(category_df.isna().sum())
print("\nDuplicated rows:", category_df.duplicated().sum())
print("\nUnique values:")
print(category_df[[
    "product_category_name",
    "product_category_name_english"
]].nunique())

print("\nProduct categories without a translation:")
print(
    products_df.loc[
        ~products_df["product_category_name"].isin(
            category_df["product_category_name"]
        ),
        "product_category_name"
    ].dropna().nunique()
)

print("\nProduct records without a translation:")
print(
    products_df.loc[
        ~products_df["product_category_name"].isin(
            category_df["product_category_name"]
        ),
        "product_category_name"
    ].dropna().shape[0]
)

print("\nTranslation categories not referenced in products:")
print(
    category_df.loc[
        ~category_df["product_category_name"].isin(
            products_df["product_category_name"]
        ),
        "product_category_name"
    ].shape[0]
)

Shape: (71, 2)

Columns: ['product_category_name', 'product_category_name_english']

Data types:
product_category_name            object
product_category_name_english    object
dtype: object

Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64

Duplicated rows: 0

Unique values:
product_category_name            71
product_category_name_english    71
dtype: int64

Product categories without a translation:
2

Product records without a translation:
13

Translation categories not referenced in products:
0


### **4. Export The Cleaned Dataset**

In [58]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [59]:
category_df.to_csv(
    CLEANED_DATA_PATH / "olist_category_cleaned.csv",
    index=False
)